# Train a network with width 400 and varying depth

In [ ]:
import pandas as pd
from DerivModel import FeedForwardNetwork, train_model
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
import torch.nn.functional as F
from tqdm.notebook import tqdm, trange
import matplotlib.pyplot as plt
%matplotlib inline
from cycler import cycler

## Load and Prepare the dataset

Here dataset A will be used with the Monte Carlo samples at 100,000 in the underlying basket option model. 

In [ ]:
filename_root = 'test_basket_mt'

In [ ]:
df1 = pd.concat(
    [pd.read_csv(f"{filename_root}{i}.csv") for i in range(20)],
    ignore_index=False
)

# Columns that need cleaning
cols_to_fix = ["maturity", "option_value", "error_estimate", "process_time", "samples"]

for col in cols_to_fix:
    if col in df1.columns:
        df1[col] = (
            df1[col]
            .astype(str)              # ensure string
            .str.strip("[]")         # remove brackets
            .astype(float)           # convert to numeric
        )

In [ ]:
df1

In [ ]:
print(df1.columns)

In [ ]:
df1['samples'].unique()

In [ ]:
df1 = df1[df1['samples'] == 100000]

In [ ]:
df1 = df1.drop('samples', axis=1)

## Train models for the supplied list of widths and 6 hidden layers

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
y = df1['option_value']

In [ ]:
X = df1.drop(['option_value', 'process_time', 'error_estimate'], axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.01) 

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
torch_tensor = torch.from_numpy(X_train).float().to(device)
y_tensor = torch.from_numpy(y_train.to_numpy().copy()).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
batch_size = 4096
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

In [ ]:
test_torch_tensor = torch.from_numpy(X_test).float().to(device)
test_y_tensor = torch.from_numpy(y_test.to_numpy().copy()).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
torch_tensor.shape

In [ ]:
# Hyperparameters
epochs = 200
lr = 0.001

In [ ]:
# Grid search parameters
layers = [2, 4, 6, 8, 10]

In [ ]:
model_out = dict()
for ilayer in layers:
    train_loss_lst = list()
    test_loss_lst = list()
        
    # Initialize model, optimizer and loss function
    model = FeedForwardNetwork(input_size=28, 
                               num_hidden_layers=ilayer, 
                               neurons_per_layer=300).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = torch.nn.MSELoss()
        
    # Train model
    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
           
        train_mse_lst = list()    
        for batch_X, batch_y in data_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            train_mse_lst.append(loss.item())
                
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        train_avg_mse = sum(train_mse_lst) / len(train_mse_lst)
        train_loss_lst.append(train_avg_mse)
        
        # Evaluate model
        model.eval()
        test_mse_list = list()
        with torch.no_grad():
            for batch_X, batch_y in test_data_loader:
                test_outputs = model(batch_X)
                mse = F.mse_loss(test_outputs.squeeze(), batch_y).item()
                test_mse_list.append(mse)
                
        test_avg_mse = sum(test_mse_list) / len(test_mse_list)
        test_loss_lst.append(test_avg_mse)
        
        tqdm_epoch.set_description(f"Depth {ilayer}: Epoch {epoch+1}/{epochs} - Train error: {train_avg_mse:.4f}, Test error: {test_avg_mse:.4f}")
    
    model_out[ilayer] = { 'train_loss': train_loss_lst, 'test_loss': test_loss_lst}
    torch.save(model.state_dict(), str(ilayer) + '_depthmodel.pt')

In [ ]:
repochs = range(1, epochs + 1)

In [ ]:
def smooth_curve(points, factor=0.9):
    smoothed_points = []
    for point in points:
        if smoothed_points:
            previous = smoothed_points[-1]
            smoothed_points.append(previous * factor + point * (1 - factor))
        else:
            smoothed_points.append(point)
    return smoothed_points

In [ ]:
fig, ax = plt.subplots(figsize=(15,7.5))

# Existing monochrome cycler
monochrome = cycler('color', ['k']) * cycler('linestyle', ['-', '--'])

# Adding a cycler for markers
marker_cycler = cycler('marker', ['+', '^', 'x', '.', 'v', 'o'])  # Example markers

# Combine the cyclers
monochrome_with_markers = marker_cycler * monochrome


ax.set_prop_cycle(monochrome_with_markers)
for key, value in model_out.items():
    ax.plot(repochs, smooth_curve(value['train_loss']), label= str(key) + ' Training Loss')
    ax.plot(repochs, smooth_curve(value['test_loss']), label=str(key) + ' Test Loss')

ymin = 0.05
ymax = 0.2
xmin = 160
xmax = 200
    
ax.set_ylim(ymin, ymax)
ax.set_xlim(xmin, xmax)
ax.set_yscale('log')
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss (log)')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
fig.savefig('BaskedOpDepth.png', dpi=300, bbox_inches='tight')